# ipadic-neologd

## 依存関係のインストール

In [ ]:
import os
from pathlib import Path


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("pyproject.toml があるディレクトリで notebook を開いてください")


PROJECT_ROOT = find_project_root()
TOOLS_DIR = PROJECT_ROOT / ".cache" / "mecab"
MECAB_PREFIX = TOOLS_DIR / "local"
NEOLOGD_DIR = TOOLS_DIR / "dic" / "mecab-ipadic-neologd"
MECAB_URL = "https://deb.debian.org/debian/pool/main/m/mecab/mecab_0.996.orig.tar.gz"
NEOLOGD_URL = "https://api.github.com/repos/neologd/mecab-ipadic-neologd/tarball/v0.0.7"

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["TOOLS_DIR"] = str(TOOLS_DIR)
os.environ["MECAB_PREFIX"] = str(MECAB_PREFIX)
os.environ["NEOLOGD_DIR"] = str(NEOLOGD_DIR)
os.environ["MECAB_URL"] = MECAB_URL
os.environ["NEOLOGD_URL"] = NEOLOGD_URL

PROJECT_ROOT


In [ ]:
%%bash
set -euxo pipefail
unset CLICOLOR_FORCE CLICOLOR LSCOLORS LS_COLORS

MECAB_TARBALL="$TOOLS_DIR/source/mecab.tar.gz"

rm -rf "$TOOLS_DIR/source/mecab-0.996" "$MECAB_PREFIX" "$MECAB_TARBALL"
mkdir -p "$TOOLS_DIR/source"

curl -fL "$MECAB_URL" -o "$MECAB_TARBALL"
tar -xzf "$MECAB_TARBALL" -C "$TOOLS_DIR/source"

cd "$TOOLS_DIR/source/mecab-0.996"
./configure --prefix="$MECAB_PREFIX" --with-charset=utf8
make -j"$(python -c 'import os; print(os.cpu_count() or 2)')"
make install


In [ ]:
%%bash
set -euxo pipefail
unset CLICOLOR_FORCE CLICOLOR LSCOLORS LS_COLORS

NEOLOGD_TARBALL="$TOOLS_DIR/source/mecab-ipadic-neologd.tar.gz"

rm -rf "$TOOLS_DIR/source/mecab-ipadic-neologd" "$NEOLOGD_DIR" "$NEOLOGD_TARBALL"
mkdir -p "$TOOLS_DIR/source/mecab-ipadic-neologd" "$TOOLS_DIR/dic"

export PATH="$MECAB_PREFIX/bin:$PATH"
curl -fL "$NEOLOGD_URL" -o "$NEOLOGD_TARBALL"
tar -xzf "$NEOLOGD_TARBALL" -C "$TOOLS_DIR/source/mecab-ipadic-neologd" --strip-components 1

cd "$TOOLS_DIR/source/mecab-ipadic-neologd"
python - <<'PY'
from pathlib import Path

script = Path("libexec/make-mecab-ipadic-neologd.sh")
text = script.read_text()
old = 'cd ${NEOLOGD_DIC_DIR}\n\nMECAB_PATH=`which mecab`'
new = 'cd ${NEOLOGD_DIC_DIR}\nfind . -type f -exec touch -t 200001010000 {} +\n\nMECAB_PATH=`which mecab`'
if old not in text:
    raise RuntimeError("patch target was not found in make-mecab-ipadic-neologd.sh")
script.write_text(text.replace(old, new, 1))
PY
./bin/install-mecab-ipadic-neologd -n -y -p "$NEOLOGD_DIR"


## 実装コード

In [28]:
import MeCab


def make_tagger() -> MeCab.Tagger:
    if not NEOLOGD_DIR.is_dir():
        raise FileNotFoundError(f"辞書がありません。install-ipadic-neologd セルを実行してください: {NEOLOGD_DIR}")

    tagger = MeCab.Tagger(f"-r /dev/null -d {NEOLOGD_DIR}")
    tagger.parse("")
    return tagger


def split_morphemes(word: str, tagger: MeCab.Tagger) -> list[dict[str, str | None]]:
    morphemes = []
    node = tagger.parseToNode(word)
    while node:
        if node.surface:
            features = node.feature.split(",")
            morphemes.append(
                {
                    "surface": node.surface,
                    "part_of_speech": features[0] if len(features) > 0 else "",
                    "base_form": features[6] if len(features) > 6 and features[6] != "*" else node.surface,
                    "reading": features[7] if len(features) > 7 and features[7] != "*" else None,
                    "pronunciation": features[8] if len(features) > 8 and features[8] != "*" else None,
                }
            )
        node = node.next

    return morphemes


## 実行

In [29]:
tagger = make_tagger()
word = "推し活"
split_morphemes(word, tagger)


[{'surface': '推し',
  'part_of_speech': '動詞',
  'base_form': '推す',
  'reading': 'オシ',
  'pronunciation': 'オシ'},
 {'surface': '活',
  'part_of_speech': '名詞',
  'base_form': '活',
  'reading': 'カツ',
  'pronunciation': 'カツ'}]